In [25]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [26]:
import os
import pandas as pd


os.chdir('/content/drive/MyDrive/AI_ML_Project_dataset')


print(os.listdir())

['personality_dataset.csv', 'dataset_cleaned.csv', 'personality_model.pkl']


In [27]:
import pandas as pd
df = pd.read_csv('dataset_cleaned.csv')
df.head()

,Time_spent_Alone,Stage_fear,Social_event_attendance,Going_outside,Drained_after_socializing,Friends_circle_size,Post_frequency,Personality
0,4.0,0.0,4.0,6.0,0.0,13.0,5.0,1
1,9.0,1.0,0.0,0.0,1.0,0.0,3.0,0
2,9.0,1.0,1.0,2.0,1.0,5.0,2.0,0
3,0.0,0.0,6.0,7.0,0.0,14.0,8.0,1
4,3.0,0.0,9.0,4.0,0.0,8.0,5.0,1


In [28]:
df.isnull().sum()

,0
Time_spent_Alone,0
Stage_fear,0
Social_event_attendance,0
Going_outside,0
Drained_after_socializing,0
Friends_circle_size,0
Post_frequency,0
Personality,0


selected 3 classification algorithms - Logistic Regression, Random Forest, and Gradient Boosting and utilizing GridSearchCV with 5-fold cross-validation to exhaustively search for the optimal hyperparameters for each model.

In [29]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

In [30]:
# Separate Features (X) and Target (y)
X = df.drop('Personality', axis = "columns")
y = df['Personality']

In [31]:
X.head()

,Time_spent_Alone,Stage_fear,Social_event_attendance,Going_outside,Drained_after_socializing,Friends_circle_size,Post_frequency
0,4.0,0.0,4.0,6.0,0.0,13.0,5.0
1,9.0,1.0,0.0,0.0,1.0,0.0,3.0
2,9.0,1.0,1.0,2.0,1.0,5.0,2.0
3,0.0,0.0,6.0,7.0,0.0,14.0,8.0
4,3.0,0.0,9.0,4.0,0.0,8.0,5.0


In [32]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [33]:
X_train.shape

(2320, 7)

In [34]:
X_test.shape

(580, 7)

In [35]:
models = {
    'Logistic Regression': (Pipeline([
        ('scaler', StandardScaler()),
        ('logreg', LogisticRegression(max_iter=1000))
    ]), {
        'logreg__C': [0.01, 0.1, 1, 10]
    }),
    'Random Forest': (RandomForestClassifier(random_state=42), {
        'n_estimators': [50, 100, 200],
        'max_depth': [None, 5, 10],
        'min_samples_split': [2, 5]
    }),
    'Gradient Boosting': (GradientBoostingClassifier(random_state=42), {
        'n_estimators': [50, 100],
        'learning_rate': [0.01, 0.1],
        'max_depth': [3, 5]
    })
}

In [36]:
best_model_name = ""
best_model_score = 0
best_model_pipeline = None

In [37]:
for name, (model, param_grid) in models.items():
    # Setup GridSearch
    grid = GridSearchCV(model, param_grid, cv=5, scoring='f1', n_jobs=-1)
    grid.fit(X_train, y_train)
    
    
    best_estimator = grid.best_estimator_
    y_pred = best_estimator.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    
    print(f"--- {name} ---")
    print(f"Best Params: {grid.best_params_}")
    print(f"Test Accuracy: {acc:.4f}")
    print(f"Test F1 Score: {f1:.4f}\n")
    
    # Track the best overall model
    if f1 > best_model_score:
        best_model_score = f1
        best_model_name = name
        best_model_pipeline = best_estimator

--- Logistic Regression ---
Best Params: {'logreg__C': 0.01}
Test Accuracy: 0.9224
Test F1 Score: 0.9236

--- Random Forest ---
Best Params: {'max_depth': 5, 'min_samples_split': 2, 'n_estimators': 50}
Test Accuracy: 0.9241
Test F1 Score: 0.9252

--- Gradient Boosting ---
Best Params: {'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 50}
Test Accuracy: 0.9224
Test F1 Score: 0.9236



In [38]:
print(f" OVERALL BEST MODEL: {best_model_name}")
print(f" WINNING F1 SCORE: {best_model_score:.4f}")

 OVERALL BEST MODEL: Random Forest
 WINNING F1 SCORE: 0.9252


In [39]:
print(classification_report(y_test, best_model_pipeline.predict(X_test)))

              precision    recall  f1-score   support

           0       0.91      0.94      0.92       282
           1       0.94      0.91      0.93       298

    accuracy                           0.92       580
   macro avg       0.92      0.92      0.92       580
weighted avg       0.92      0.92      0.92       580



Precision (Quality of the Guesses):

Extrovert (0.94): When the AI predicts someone is an Extrovert, it is correct 94% of the time. It very rarely falsely labels an Introvert as an Extrovert.

Introvert (0.91): When it predicts Introvert, it is correct 91% of the time.

Recall (Catching all of them):

Introvert (0.94): Out of all the actual Introverts in the test set, the AI successfully found 94% of them. It only missed 6% (classifying them as Extroverts by mistake).

Extrovert (0.91): It successfully caught 91% of the actual Extroverts.

Checking for overfitting

In [40]:
best_model_pipeline

RandomForestClassifier(max_depth=5, n_estimators=50, random_state=42)

In [41]:

final_model = best_model_pipeline

# Predict on the Training Data
train_preds = final_model.predict(X_train)
train_f1 = f1_score(y_train, train_preds)
train_acc = accuracy_score(y_train, train_preds)

In [42]:
# Predict on the Testing Data
test_preds = final_model.predict(X_test)
test_f1 = f1_score(y_test, test_preds)
test_acc = accuracy_score(y_test, test_preds)

In [43]:
print("Overfitting Check")
print(f"Training F1-Score: {train_f1:.4f}  |  Training Accuracy: {train_acc:.4f}")
print(f"Testing F1-Score:  {test_f1:.4f}  |  Testing Accuracy:  {test_acc:.4f}")

Overfitting Check
Training F1-Score: 0.9382  |  Training Accuracy: 0.9371
Testing F1-Score:  0.9252  |  Testing Accuracy:  0.9241


In [44]:
# Calculate the drop in performance
drop = (train_f1 - test_f1) * 100
print(f"\nPerformance drop on unseen data: {drop:.2f}%")


Performance drop on unseen data: 1.30%


drop of about 1.30%. Anything under a 5% drop is considered elite, production-grade generalization!

    CONCLUSION: So good model and no overfitted

In [45]:
import joblib

# Export the model to a file
model_filename = 'personality_model.pkl'
joblib.dump(best_model_pipeline, model_filename)

print(f"Success! Model saved as {model_filename}")

Success! Model saved as personality_model.pkl


In [46]:
# This automatically scales the data (Mean=0, Variance=1) BEFORE feeding it to Logistic Regression

production_pipeline = Pipeline([

    ('scaler', StandardScaler()),

    ('logreg', LogisticRegression(C=0.01, max_iter=1000, random_state=42))

])

# Train the Pipeline

print("Training the Standardized Pipeline...")

production_pipeline.fit(X_train, y_train)

Training the Standardized Pipeline...


Pipeline(steps=[('scaler', StandardScaler()),
                ('logreg',
                 LogisticRegression(C=0.01, max_iter=1000, random_state=42))])

In [47]:
#Evaluate the Pipeline

y_pred = production_pipeline.predict(X_test)

print("\n--- Final Standardized Model Performance ---")

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")

print(f"F1-Score: {f1_score(y_test, y_pred):.4f}")


--- Final Standardized Model Performance ---
Accuracy: 0.9224
F1-Score: 0.9236


In [48]:
#Export the entire Pipeline

model_filename = 'personality_pipeline.pkl'

joblib.dump(production_pipeline, model_filename)



print(f"\n✅ Success! Standardized Pipeline saved as {model_filename}")


✅ Success! Standardized Pipeline saved as personality_pipeline.pkl


Occam's Razor: If two models perform equally well, always choose the simpler one. so I chose Logistic Regession